In [1]:
import numpy as np
import pandas as pd

trans_df = pd.read_csv("../datasets/HI-Small_Trans.csv")
trans_df.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [2]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/18 16:18]


In [3]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Account
80A7FD400    2
80A7FDE00    2
80FA55EF0    2
80FA56340    2
81211BA20    2
81211BC00    2
8135B8200    2
8135B8250    2
Name: Bank, dtype: int64

In [4]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 1895172


In [5]:
# Analyze accounts.
accounts_df = pd.read_csv("../datasets/HI-Small_accounts.csv")
print("SIZE:", accounts_df.shape[0])
accounts_df.head(5)

SIZE: 518581


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [6]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 1032664


In [7]:
def filter_function(x):
    unique_account_size = x.groupby(["To Bank", "Account.1"]).size().size
    return  unique_account_size > 5 and unique_account_size < 10
ranged_trans_usd_sept_df = trans_usd_sept_1st_df.groupby(["From Bank", "Account"]).filter(filter_function)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 165460


In [8]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions

,From Bank,Account,To Bank,Account.1,Amount Paid
1,3208,8000F4580,1,8000F5340,0.01
6,1,8000EBAC0,1,8000EBAC0,14.26
7,1,8000EC1E0,1,8000EC1E0,11.86
8,12,8000EC280,2439,8017BF800,7.66
10,1,8000F4510,11813,8011305D0,9.82
...,...,...,...,...,...
5075878,18679,807951C10,253359,814901FB0,9.30
5075884,220,8001357A0,220,8001357A0,17.65
5076627,2439,80343CDA0,2439,80343CDA0,49.32
5078316,215064,808F06E11,215064,808F06E10,0.07


In [9]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]]

,From Bank,Account,Bank Name,Amount Paid
0,1,801ABB8F0,Arbor Savings Bank,1.662061e+10
1,1,801ABB8F0,Arbor Savings Bank,1.662061e+10
2,1,801ABB8F0,Arbor Savings Bank,1.662061e+10
3,1,801ABB8F0,Arbor Savings Bank,1.662061e+10
4,1,801ABB8F0,Arbor Savings Bank,1.662061e+10
...,...,...,...,...
384959,356096,814829A20,Bank of Billings,1.603500e+02
384960,356097,81482BAC0,First Bank of Houston,2.199300e+02
384961,356172,8148A7740,National Bank of Cleveland,4.600800e+02
384962,356215,814900780,Savings Bank of New York,4.766550e+04


In [10]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]]

,From Bank,Account,Payment Format,Amount Paid
0,241869,80F390FC0,ACH,6240.05
1,241869,80F390FC0,ACH,7782.25
3,211,808E44B10,ACH,1589.02
5,211,808E44B10,ACH,933.63
6,211,808E44B10,ACH,8298.19
...,...,...,...,...
862448,23537,803949A90,ACH,7823.96
862449,16163,803638A90,ACH,12667.62
862450,16163,803638A90,ACH,3020.41
862451,215064,808F06E11,ACH,0.07


In [11]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
account_pairs_df = accounts_df.merge(accounts_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts

,Bank,Account
0,1,8000555D0
26,1,800056370
39,1,800057A10
51,1,80005C0A0
65,1,80005C3D0
...,...,...
36602,38132,80E2D2440
36615,111433,80D92DE40
36617,15916,80544C020
36620,218472,810E94D40


In [12]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
#Map everything to US Dollars by a fixed table.
TO_US_DOLLARS = {
    "Australian Dollar": 0.72,
    "Bitcoin": 78.33,
    "Brazil Real": 0.20,
    "Canadian Dollar": 0.73,
    "Euro": 1.17,
    "Mexican Peso": 0.06,
    "Ruble": 0.01,
    "Rupee": 0.01,
    "Saudi Riyal":0.27,
    "Shekel": 0.33,
    "Swiss Franc": 1.27,
    "UK Pound":  1.35,
    "US Dollar": 1,
    "Yen": 0.006,
    "Yuan": 0.15
}
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df['Amount Paid'] * trans_sept_1st_wire_or_ach_converted_df['Payment Currency'].map(TO_US_DOLLARS)
print("SIZE:", trans_sept_1st_wire_or_ach_converted_df.shape[0])

SIZE: 392128
